In [2]:
#Task 4.1
import sqlite3
import csv

script = """
    CREATE TABLE IF NOT EXISTS User(
        user_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT NOT NULL,
        hash TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS Credit(
        dt_transaction DATETIME NOT NULL,
        type TEXT NOT NULL,
        amt DECIMAL NOT NULL,

        user_id INT NOT NULL,
        FOREIGN KEY (user_id) REFERENCES User(user_id) ON DELETE CASCADE
    );
"""

conn = sqlite3.connect("USER_DATA.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table'")
tables = cursor.fetchall()

for table in tables:
    cursor.execute(f'DROP TABLE IF EXISTS "{table[0]}";')

cursor.executescript(script)

with open("./Resource Files/users.txt") as f:
    data = [i for i in csv.reader(f)][1:]
    
    for i in data:
        cursor.execute("INSERT INTO User (user_id, name, email, hash) VALUES (?, ?, ? ,?)", (i[0], i[1], i[2], i[3]))

with open("./Resource Files/credits.txt") as f:
    data = [i for i in csv.reader(f)][1:]

    for i in data:
        cursor.execute("INSERT INTO Credit (dt_transaction, type, amt, user_id) VALUES (?, ?, ? ,?)", (i[1], i[2], i[3], i[0]))

conn.commit()
conn.close()

In [3]:
#Task 4.2
def validate_card(card_number):
    card_number = str(card_number)

    checkSum = 0
    for i in range(len(card_number)-1, -1, -1):
        if i % 2 == 0:
            double = str(int(card_number[i]) * 2)

            if len(double) == 2:
                checkSum+=int(double[0]) + int(double[1])
            else:
                checkSum+=int(double)
        else:
            checkSum+=int(card_number[i])

    return checkSum % 10 == 0

print(validate_card(4417123456789113))

print(validate_card(4532840913491054))
print(validate_card(45555734352999785))
print(validate_card(378282904122908))
print(validate_card(4111025640404402))
print(validate_card(5105207030955616))

True
True
False
False
False
False


In [4]:
#Task 4.3
def check_user(user_id):
    conn = sqlite3.connect("USER_DATA.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM User WHERE user_id = ?", (user_id,))
    info = cursor.fetchone()

    conn.close()

    return info != None

check_user("among-us")
check_user("meow")

False

In [5]:
#Task 4.4
def SHA3hash(string):
    import hashlib
    sha3_hash = hashlib.sha3_256(string.encode()).hexdigest()

    return sha3_hash

def update_card(user_id, new_num):
    if not check_user(user_id): return False
    if not validate_card(new_num): return False

    salt = "@SR_SUP3R_S3CURE_S4LT!"
    hashed = SHA3hash(salt+str(new_num))

    conn = sqlite3.connect("USER_DATA.db")
    cursor = conn.cursor()

    cursor.execute("UPDATE User SET hash = ? WHERE user_id = ?", (hashed, user_id))

    conn.commit()
    conn.close()
    return True

print(update_card("swimmerz", "30569309025904")) 
print(update_card("koh-world", "1122334455667788"))

True
False


In [15]:
#Task 4.5
from flask import Flask, request, render_template
import sqlite3
from datetime import datetime

curr_day = datetime(2025, 8, 30, 0, 0, 0)

app = Flask(__name__, template_folder='TASK_4_5')

@app.route('/', methods=['POST', 'GET'])
def base():
    name = request.form.get('name')

    conn = sqlite3.connect('USER_DATA.db')
    cursor = conn.cursor()

    cursor.execute('SELECT user_id FROM User WHERE name = ?', (name,))
    user_id = cursor.fetchone()
    if not user_id:
        return render_template('base.html')
    user_id = user_id[0]

    cursor.execute('SELECT * FROM Credit WHERE user_id = ?', (user_id,))
    transactions = cursor.fetchall()
    transactions = [i for i in transactions if (curr_day - datetime.strptime(i[0], "%Y-%m-%d %H:%M:%S")).days <= 90]

    conn.close()

    return render_template('base.html', name=name, transactions=transactions)

app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [08/Apr/2026 11:28:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 11:28:27] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 11:28:49] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 11:29:02] "POST / HTTP/1.1" 200 -
